## TIFON Databases. Read and load example databases

In [1]:
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

def read_db(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    db.extract_inputs(
        id_groups = (3,),
        cases_idx = case_idx,
        vtu_type='surface',
        verbose=False
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=0, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=1, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )
    
    return db

0 Warning! Import - NVTX not present!
/home/m.jaraiz/miniconda/envs/envkan_amd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Base de datos original
case_idx = list(range(100))
fuera = [64, 79, 87, 88, 94]
for c in fuera:
    case_idx.remove(c)
db_0 = read_db(datafolder='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3/', case_idx=case_idx)


 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3
100 simulations found.
Parse taked: 0.1090 seconds


In [3]:
# Base de datos adicional
db_1 = read_db('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_prueba/', case_idx='all')


 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_prueba
5 simulations found.
Parse taked: 0.0085 seconds


In [4]:
print(db_0.metadata['design_vars'])
print(db_1.metadata['design_vars'])
flcc = db_1.data_dict['CADGroup_3']['FlCc'][:, :-1]
design_vars = db_1.metadata['design_vars']
db_1.metadata['design_vars'] = design_vars[:-1]
db_1.data_dict['CADGroup_3']['FlCc'] = flcc
print(db_0.metadata['design_vars'])
print(db_1.metadata['design_vars'])

['aoa', 'mach']
['aoa', 'mach', 'h']
['aoa', 'mach']
['aoa', 'mach']


In [5]:
for db in [db_0, db_1]:
    print(db.data_dict['CADGroup_3']['FlCc'].shape)

(95, 2)
(5, 2)


### Merge two sets

In [6]:
db_completo = FRODO.merge_datasets(
    root_dir='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed',
    sources = [(db_0, '3'), (db_1, '3')],
    new_group_id='3_completo',
    k=4,
    mesh_ref=0,
    cache=True,
    get_df_metrics_attr={
        'var_metrics': ['CoefLift', 'CoefDrag', 'CoefMomentY'],
        'iter_var': 1000,
        'save' : False
    }
    
)
db_completo.summary_data()

AttributeError: 'FRODO' object has no attribute 'kwargs'

In [ ]:
import pandas as pd
df_post = pd.read_csv(
    '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed/metadata/df_post.csv',
    sep=',',
    header=0,
)
df_post.sort_values(by=['aoa', 'mach'], inplace=True)

In [ ]:
d = []
for stage, factor in zip([0, 1], [1, 1]):
    vars = {
        'aoa' : {'idim':0, 'value':df_post['aoa'].values}, # Angle of attack
        'M'   : {'idim':0, 'value':df_post['mach'].values},   # Mach number
        'Re'  : {'idim':0, 'value':df_post['re'].values},  # Reynolds
        'CL'  : {'idim':0, 'value':df_post[f'coeflift_mean_stage{stage}'].values / factor},  # Mean Lift coefficient
        'CD'  : {'idim':0, 'value':df_post[f'coefdrag_mean_stage{stage}'].values / factor},  # Mean Drag coefficient
        'CMy'  : {'idim':0, 'value':df_post[f'coefmomenty_mean_stage{stage}'].values / factor},  # Mean Momentum coefficient
        'varCL'  : {'idim':0, 'value':df_post[f'coeflift_var_stage{stage}'].values},  # Var Lift coefficient
        'varCD'  : {'idim':0, 'value':df_post[f'coefdrag_var_stage{stage}'].values},  # Var Drag coefficient
        'varCM'  : {'idim':0, 'value':df_post[f'coefmomenty_var_stage{stage}'].values},  # Var Momentum coefficient
    }
    d.append(db_completo.sets.create_NN_pylom(id_groups=['3_completo',], stage=stage, idx_to_print='all', external_vars=vars, save_path='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed/outputs/'))

### Interpolation between meshes

In [ ]:
new_mesh = {
    "Coord": db_1.data_dict['CADGroup_3']['Coord'],
    "Conec": db_1.data_dict['CADGroup_3']['Conec'],
    "NodeCoord": db_1.data_dict['CADGroup_3']['NodeCoord'],
    # lo que tengas
}
db_0.sets.create_group_by_interpolation(
    id_group_src='3',
    new_group_id='3_interp',
    new_mesh=new_mesh,
    method='idw',
    k=8
    )

In [ ]:
db_0.summary_data()